# 03 · Oxygen track: SpO₂ concern rule on breath-hold data

A **deterministic concern rule** for SpO₂. It is the oxygen half of the two-track design, kept separate from the heart-rate track (notebook 04). Nothing is trained on outcome labels, because no open dataset pairs SpO₂ with observed blackouts. The firmware runs the `concern_level` variant with the deep anchors (80 → 60%).

| | |
|---|---|
| **Main data** | [Physiological and autonomic responses to intermittent low-lung-volume apnoea](https://zenodo.org/records/21701010) (CC BY 4.0): 80 healthy adults lying supine on land for six minutes, with Nonin finger SpO₂ and Polar H10 heart rate at 1 Hz. 40 people (group **AG**) perform repeated breath-holds; 40 (group **CG**) breathe normally. Each person's **resting SpO₂ was recorded before the session**; this is the independently measured baseline. All 80 stayed conscious; the only reported adverse effects were one headache and one light dizziness. |
| **Deep desaturations** | [UW/UCSD controlled hypoxemia](https://github.com/ubicomplab/oximetry-phone-cam-data) (MIT): 6 awake adults, four clinical finger oximeters at 1 Hz, lowest readings 61–77% (Evaluation 5). |
| **Labels** | Only (a) protocol group membership (breath-hold vs normal breathing, assigned by the study) and (b) the fact that nobody lost consciousness, which makes every alert a non-blackout alert. Same-stream "engineering events" and synthetic crossings are timing references, not outcomes, and are named as such. |
| **Method** | Six rule variants and two fitted Kalman variants, replayed causally second by second under three anchor sets, then five evaluations. Every threshold and timing constant is fixed in the setup cell and evaluated, not tuned. |
| **Outputs** | Configuration (`deployable: false`), result tables, figures and an importable `replay_reference.py` in `outputs/oxygen_concern/`. |

> **Limits.** The constants are **not pre-registered**: earlier versions of this notebook were run on the same data, and their results informed this version. The dry breath-holds go no lower than 83%, so only UW/UCSD exercises the deep anchors, and its reference is the other finger oximeters, not arterial blood. Version history is in the [folder README](README.md#version-history).

**What this notebook can and cannot show**

| Question | Answerable here? | How |
|---|---|---|
| Does the rule respond to breath-hold desaturation, using labels that do not come from SpO₂? | Yes | AG vs CG group membership is set by the protocol |
| How often does it alert during *safe* breath-holds and normal breathing? | Yes | No one blacked out, so every alert is a non-blackout alert |
| How fast does it react once SpO₂ crosses a level? | Engineering timing only | Same-stream crossings: labelled as such, not accuracy |
| Is it robust to sensor noise and dropouts? | Yes, by simulation | Seeded perturbation of the real traces |
| How does it behave on **real** desaturations to 61–77%, timed against independent oximeters? | Yes (Evaluation 5) | UW/UCSD: four clinical oximeters; rule on one device, reference = median of the other three |
| Does it alert **before blackout**? | **No** | The datasets used here have no blackout outcomes. Dry breath-holds reach 83%; UW/UCSD supplies real deeper desaturations but no blackout timing labels. |

## How to run

On Google Colab, keep the default CPU runtime and choose **Runtime → Run all**. Authorise Google Drive when asked: outputs are saved to `MyDrive/KappaBetaML/outputs/oxygen_concern`. The notebook downloads one checksum-pinned 0.7 MB workbook and six small checksum-pinned UW/UCSD CSV files.

| Setting | Effect |
|---|---|
| `SAVE_TO_DRIVE = False` (`KB_SAVE_TO_DRIVE=0`) | Keep everything in the runtime instead of Drive |
| `CONCERN_DATA_ROOT` | Folder searched first for unchanged originals (default `MyDrive/KappaBetaML/input/`), e.g. `Supp database.xlsx` |
| `ALLOW_DOWNLOAD = False` (`CONCERN_DOWNLOAD=0`) | Never download; use attached or cached files only |
| `CONCERN_CACHE`, `CONCERN_OUTPUT_DIR` | Download cache and output folder |
| `KB_OPENOX=1` | Run the optional restricted OpenOximetry section (off by default; needs your PhysioNet login) |

## Setup

Package check, imports and statistics helper, then storage, then every rule and evaluation constant.

In [ ]:
import importlib.util

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "openpyxl": "openpyxl",
    "requests": "requests",
    "matplotlib": "matplotlib",
    "IPython": "ipython",
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    raise ImportError(
        "Missing packages: "
        + ", ".join(missing)
        + ". Install only these in a separate %pip install cell, then restart and Run All."
    )
from pathlib import Path
import hashlib, json, math, os, platform, re, statistics, time
import numpy as np
import pandas as pd
import openpyxl, requests
from scipy import stats
import matplotlib.pyplot as plt
from IPython.display import display


def env_flag(name, default):
    return os.getenv(name, str(default)).lower() in {"1", "true", "yes"}


def poisson_rate_ci(count, hours, level=0.95):
    """Exact (Garwood) Poisson CI for an event rate per hour. Treats alert starts as independent, which
    understates the width when a few people produce most alerts."""
    a = 1 - level
    lo = 0.0 if count == 0 else stats.chi2.ppf(a / 2, 2 * count) / 2
    hi = stats.chi2.ppf(1 - a / 2, 2 * count + 2) / 2
    return lo / hours, hi / hours

### Storage

In [ ]:
# ---- Storage: Google Colab ----
# Colab's /content disk is wiped when the runtime ends, so outputs (and the downloads cache for
# small sources) go to Google Drive by default. Set SAVE_TO_DRIVE = False to keep everything in this
# runtime.
import sys

ON_COLAB = "google.colab" in sys.modules
SAVE_TO_DRIVE = env_flag("KB_SAVE_TO_DRIVE", ON_COLAB)
if ON_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/KappaBetaML")
else:
    BASE = Path("/content") if ON_COLAB else Path(".")
# optional manually uploaded originals
DATA_ROOT = Path(os.getenv("CONCERN_DATA_ROOT", str(BASE / "input")))
OUT = Path(os.getenv("CONCERN_OUTPUT_DIR", str(BASE / "outputs" / "oxygen_concern")))
FIGURES = OUT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
CACHE = Path(os.getenv("CONCERN_CACHE", str(BASE / "cache" / "oxygen_concern")))
ALLOW_DOWNLOAD = env_flag("CONCERN_DOWNLOAD", True)
SEED = 2026

### Rule and evaluation constants

In [ ]:
# ---- Rule constants (fixed, not tuned, not pre-registered; not validated alarm limits) ----
# Anchor sets map SpO2 onto a 0-1 concern scale: 0 at `start`, 1 at `full`.
ANCHOR_SETS = {
    "clinical_hypoxemia": {"start": 95.0, "full": 85.0},  # conventional clinical range
    "moderate": {"start": 90.0, "full": 75.0},
    # freedivers often surface at 55-80% conscious
    "deep_breath_hold": {"start": 80.0, "full": 60.0},
}
PRIMARY_ANCHORS = "deep_breath_hold"  # the set intended for swimming; see README for rationale
ENTER_SCORE, EXIT_SCORE = 0.50, 0.25  # hysteresis on the 0-1 concern index
SMOOTH_S = 3  # rolling median of 1 Hz SpO2 (device reports one value per second)
TAU_S = 3.0  # exponential filter on concern, seconds
CONFIRM_S = 3  # concern must stay >= ENTER for this long before an alert opens
RECOVER_S = 5  # concern must stay <= EXIT for this long before an alert clears
# Person-relative branch: the same anchor set, shifted up by how far the verified resting SpO2 sits
# above a reference resting level (healthy sea-level resting SpO2 is typically 95-99%; 97%
# declared). It is combined with the absolute map by max(), so it only adds sensitivity for people
# with a high baseline, and it can never fire shallower than the anchor set plus (baseline -
# reference).
REFERENCE_RESTING_SPO2 = 97.0
TREND_HORIZON_S = 15.0  # trend branch projects SpO2 this far ahead
ALPHA, BETA = 0.5, 0.1  # alpha-beta (steady-state Kalman, constant-slope) tracker gains
# cumulative-deficit rule: alert after (start-full)/2 pp sustained ~DOSE_SECONDS
DOSE_SECONDS = 10.0
ENGINEERING_EVENT_S = 5  # same-stream timing events: at/below anchor midpoint for >= 5 elapsed s
# On the device an open alert never clears by itself: a person must acknowledge it
# (replay(auto_clear=False)).
# The burden evaluations count alert *starts* with auto_clear=True, i.e. assuming an acknowledgement
# as soon as the reading has recovered for RECOVER_S; that is the most alerts a lifeguard could be
# asked to answer.

# Synthetic-test sensor model (assumptions, swept as a sensitivity analysis; not measured for the
# chest MAX30102).
# transport/response delay from arterial blood to the reported reading
SENSOR_DELAY_S = [0, 10, 20, 40]
# the Maxim routine averages a 4-s window and is recomputed every second
DEVICE_WINDOW_S = 4
# Burst-artefact stress test. When motion noise dominates both PPG channels the red/IR ratio R tends
# to 1; the firmware's Maxim calibration (SpO2 = -45.060 R^2 + 30.354 R + 94.845) maps R = 1 to
# 80.1%.
ARTIFACT_SPO2, ARTIFACT_SD = 80.1, 2.0
ARTIFACT_FRACTION, ARTIFACT_MEAN_BURST_S = 0.05, 8.0  # assumed: 5% of time in bursts averaging 8 s
assert (
    all(a["start"] > a["full"] for a in ANCHOR_SETS.values()) and 0 <= EXIT_SCORE < ENTER_SCORE <= 1
)
print("Primary anchors:", PRIMARY_ANCHORS, ANCHOR_SETS[PRIMARY_ANCHORS], "| output:", OUT.resolve())

## Acquire and verify the breath-hold workbook

The loader searches attached inputs first, then the cache, then downloads the pinned original. An MD5 mismatch stops the run: a changed source needs review, not silent substitution.

Rows 81–120 of the workbook are unused template rows and are excluded by requiring group `AG` or `CG`. The 360 one-second columns are the actual SpO₂ (`SpO2` sheet) and HR (`FC` sheet) time series.

In [ ]:
SOURCE = (
    "Supp database.xlsx",
    "https://zenodo.org/records/21701010/files/Supp%20database.xlsx?download=1",
    "79fc9abbd6e3e04c52d17df3247751ec",
)


def md5_ok(path, expected):
    return hashlib.md5(Path(path).read_bytes()).hexdigest() == expected


def acquire(name, url, md5):
    candidates = [p for p in DATA_ROOT.rglob(name)] if DATA_ROOT.is_dir() else []
    candidates.append(CACHE / name)
    for path in candidates:
        if path.is_file() and md5_ok(path, md5):
            return path
    if not ALLOW_DOWNLOAD:
        raise FileNotFoundError(f"Attach the original {name} below {DATA_ROOT} or enable Internet.")
    CACHE.mkdir(parents=True, exist_ok=True)
    part = CACHE / (name + ".part")
    for attempt in range(3):
        try:
            response = requests.get(
                url, timeout=(12, 90), headers={"User-Agent": "breath-hold-concern-notebook/2.0"}
            )
            response.raise_for_status()
            part.write_bytes(response.content)
            if not md5_ok(part, md5):
                raise ValueError("Checksum mismatch: the source changed; review before use.")
            part.replace(CACHE / name)
            return CACHE / name
        except (requests.RequestException, OSError, ValueError) as error:
            last = error
            time.sleep(1 + attempt)
    raise RuntimeError(f"Cannot acquire {name}: {last}")


workbook_path = acquire(*SOURCE)

Read the 1 Hz SpO₂ (`SpO2` sheet) and HR (`FC` sheet) series and each person's resting values (`STA_BVP` sheet).

In [ ]:
def time_series(workbook, sheet):
    rows = workbook[sheet].iter_rows(values_only=True)
    header = next(rows)
    columns = [i for i, h in enumerate(header) if re.fullmatch(r"\d{1,2}:\d{2}:\d{2}", str(h))]
    if [int(pd.to_timedelta(str(header[i])).total_seconds()) for i in columns] != list(range(360)):
        raise ValueError(f"Unexpected time columns in {sheet}")
    return {
        int(r[0]): (r[1], np.array([np.nan if r[i] is None else float(r[i]) for i in columns]))
        for r in rows
        if r[1] in {"AG", "CG"} and r[0] is not None
    }


workbook = openpyxl.load_workbook(workbook_path, read_only=True, data_only=True)
try:
    spo2_rows, hr_rows = time_series(workbook, "SpO2"), time_series(workbook, "FC")
    rows = workbook["STA_BVP"].iter_rows(values_only=True)
    header = list(next(rows))
    col = {
        name: header.index(name)
        for name in [
            "ID",
            "Group (AG or CG)",
            "Resting_SpO2",
            "Resting_HR (bpm)",
            "Apnoeist (YES o NO)",
            "Adverse_effects",
        ]
    }
    meta = pd.DataFrame(
        [
            {k: r[i] for k, i in col.items()}
            for r in rows
            if r[col["Group (AG or CG)"]] in {"AG", "CG"}
        ]
    )
finally:
    workbook.close()
meta = meta.rename(
    columns={
        "ID": "id",
        "Group (AG or CG)": "group",
        "Resting_SpO2": "resting_spo2",
        "Resting_HR (bpm)": "resting_hr",
        "Apnoeist (YES o NO)": "apnoeist",
        "Adverse_effects": "adverse_effects",
    }
).set_index("id")
assert (
    spo2_rows.keys() == hr_rows.keys() == set(meta.index)
), "Participant IDs differ between sheets"

Build one session per participant (out-of-range readings become missing) and summarise the data by group.

In [ ]:
sessions = []
for pid in sorted(spo2_rows):
    group, oxygen = spo2_rows[pid]
    oxygen = np.where((oxygen > 0) & (oxygen <= 100), oxygen, np.nan)
    baseline = pd.to_numeric(meta.loc[pid, "resting_spo2"], errors="coerce")
    sessions.append(
        {
            "subject": f"dry_apnea/{pid}",
            "group": group,
            "t": np.arange(360.0),
            "oxygen": oxygen,
            "baseline": float(baseline) if 0 < baseline <= 100 else None,
        }
    )
audit = pd.DataFrame(
    [
        {
            "subject": s["subject"],
            "group": s["group"],
            "valid_seconds": int(np.isfinite(s["oxygen"]).sum()),
            "min_spo2": np.nanmin(s["oxygen"]),
            "resting_baseline": s["baseline"],
        }
        for s in sessions
    ]
)
print(
    audit.groupby("group")
    .agg(
        people=("subject", "size"),
        valid_hours=("valid_seconds", lambda x: x.sum() / 3600),
        median_min_spo2=("min_spo2", "median"),
        lowest_spo2=("min_spo2", "min"),
        baselines=("resting_baseline", "count"),
    )
    .to_string()
)
print("Adverse effects reported:", meta.adverse_effects.value_counts().to_dict())
print(
    "Breath-hold desaturation in this study is mild: compare the lowest SpO2 with the anchor sets above."
)

## Anchor check and the Kalman measurement filter

First, the primary start anchor is re-derived from the data: the lowest breath-hold SpO₂ minus an assumed 3-point margin. It must equal the declared value, or the run stops. The margin is an engineering assumption, not a measured MAX30102 error bound.

Then the Kalman filter is fitted. Q and R describe a linear-Gaussian measurement model. A good likelihood does not establish physiological accuracy. Calibration and comparison participants remain visible in exports.

In [ ]:
from scipy.optimize import minimize

# No outcome labels: fit a linear-Gaussian measurement model to NORMAL-BREATHING recordings only.
# Fixed split by original ID, before fitting. All participants were inspected in earlier
# development.
cg = sorted(
    [s for s in sessions if s["group"] == "CG"], key=lambda s: int(s["subject"].split("/")[-1])
)
KALMAN_FIT_IDS = [s["subject"] for s in cg[::2]]
KALMAN_CHECK_IDS = [s["subject"] for s in cg[1::2]]
KALMAN_FIT_SERIES = [s["oxygen"] for s in cg[::2]]
# Engineering margin, NOT measured MAX30102 accuracy or a 95% error bound (especially below 90%).
ANCHOR_MARGIN_PP = 3.0
safe_observed_min = float(audit.loc[audit.group.eq("AG"), "min_spo2"].min())
DERIVED_ANCHORS = {"start": safe_observed_min - ANCHOR_MARGIN_PP, "full": 60.0}
assert (
    DERIVED_ANCHORS == ANCHOR_SETS[PRIMARY_ANCHORS]
), "New data changed the rationale: review anchors explicitly"
ANCHOR_AUDIT = {
    "observed_AG_minimum": safe_observed_min,
    "margin_pp_assumption": ANCHOR_MARGIN_PP,
    "start": DERIVED_ANCHORS["start"],
    "full": DERIVED_ANCHORS["full"],
    "full_anchor_basis": "inherited engineering value; not inferred from these healthy outcomes",
    "calibration_status": "inspected dataset; no validated alarm limit or measured device error bound",
}

### Kalman level/slope filter

Q and R minimise the one-step innovation likelihood on every second normal-breathing participant; the others are only scored.

In [ ]:
def kalman_step(level, slope, p00, p01, p11, reading, q, r):
    # State=[level, slope], dt=1 second. Continuous white acceleration Q=q*[[1/3,1/2],[1/2,1]].
    a, b, c = p00 + 2 * p01 + p11 + q / 3, p01 + p11 + q / 2, p11 + q
    innovation = reading - level - slope
    variance = a + r
    k0, k1 = a / variance, b / variance
    return (
        level + slope + k0 * innovation,
        slope + k1 * innovation,
        (1 - k0) * a,
        (1 - k0) * b,
        c - k1 * b,
        innovation,
        variance,
    )


def innovation_nll(log_parameters, series):
    q, r = np.exp(log_parameters)
    total, count = 0.0, 0
    for values in series:
        level = None
        for reading in values:
            if not np.isfinite(reading):
                level = None
                continue
            if level is None:
                level, slope, p00, p01, p11 = float(reading), 0.0, r, 0.0, 1.0
                continue
            level, slope, p00, p01, p11, e, v = kalman_step(
                level, slope, p00, p01, p11, reading, q, r
            )
            total += 0.5 * (np.log(2 * np.pi * v) + e * e / v)
            count += 1
    return total / max(count, 1)


fit_result = minimize(
    innovation_nll,
    np.log([0.01, 1.0]),
    args=(KALMAN_FIT_SERIES,),
    method="L-BFGS-B",
    bounds=[(np.log(1e-6), np.log(10.0)), (np.log(1 / 12), np.log(25.0))],
)
if not fit_result.success:
    raise RuntimeError(f"Kalman likelihood fit failed: {fit_result.message}")
KALMAN_Q, KALMAN_R = map(float, np.exp(fit_result.x))
KALMAN_FIT = {
    "q": KALMAN_Q,
    "r": KALMAN_R,
    "fit_subjects": KALMAN_FIT_IDS,
    "check_subjects": KALMAN_CHECK_IDS,
    "objective": "Gaussian one-step innovation negative log likelihood, no clinical outcomes",
    "train_nll": float(fit_result.fun),
    "check_nll": float(innovation_nll(fit_result.x, [s["oxygen"] for s in cg[1::2]])),
    "bounds_q": [1e-6, 10.0],
    "bounds_r": [1 / 12, 25.0],
    "r_at_lower_bound": bool(np.isclose(KALMAN_R, 1 / 12)),
    "inspected_split": True,
    "note": "Q/R separation relies on this state-space model, not a measured true SpO2 reference",
}

### Known processing delay

The only delay that `kalman_delay` compensates. Unknown physiological and oximeter delays are never estimated away.

In [ ]:
# 100 averaged samples represent 400 raw samples at 100 Hz: centre age is 1.995 s.
# This includes the four-sample decimator's 0.015 s age. It is a nominal input-window
# centre, not a measured group delay of the nonlinear Maxim estimator or circulation.
DEVICE_PROCESSING_DELAY_S = (400 - 1) / (2 * 100)
# Clinical oximeter processing delays are not known here: project ZERO seconds on their measured
# streams.
# Synthetic four-point 1-Hz trailing mean has known centre age (4-1)/2 = 1.5 s; use that exact delay
# there.
print("Anchor audit:", ANCHOR_AUDIT)
print("Fitted Kalman model:", KALMAN_FIT)

## The concern rule and the comparison variants

All variants run causally, one second at a time, on the same 1 Hz SpO₂ the wearable reports. Missing or discontinuous seconds never count as reassurance. An open alert stays latched across a gap, and recovery needs new valid observations.

| Variant | Idea |
|---|---|
| `threshold` | Alert immediately when smoothed SpO₂ ≤ anchor midpoint |
| `threshold_persist` | Same, with `CONFIRM_S` confirmation and `RECOVER_S` recovery |
| `concern_level` | 0–1 concern from the anchor map, exponential filter, hysteresis, confirmation and recovery |
| `concern_baseline` | `concern_level`, plus the same anchor map **shifted by the person's verified resting SpO₂** relative to `REFERENCE_RESTING_SPO2` (max of the two) |
| `concern_trend` | `concern_baseline`, plus an alpha-beta (steady-state Kalman) level/slope tracker that projects SpO₂ `TREND_HORIZON_S` ahead |
| `deficit_dose` | Cumulative oxygen deficit below `start` (CUSUM-style hypoxic dose) |

The concern index is a designed severity scale, **not a probability**. The concern variants share one state machine, so differences come only from their inputs.

**Clearing.** On the device (`auto_clear=False`) an open alert stays on until a person acknowledges it. A reading that *looks* normal must not cancel a drowning alarm, because motion artefacts can produce plausible values. The evaluations below replay with `auto_clear=True`: an alert clears after `RECOVER_S` seconds of recovered readings. This counts every separate alert a lifeguard would have to answer.

**Reading `replay`.** The firmware rule is `concern_level`. In `replay`, follow the final `else` branch (the concern family) with `baseline` and `trend` both off. The rest of the loop is shared: gap handling, the 3-s median, the alpha-beta tracker (used only by `concern_trend`) and the alert state machine.

In [ ]:
def ramp(u):
    # Linear 0-1 ramp; clipped. Kept linear so every anchor has a direct SpO2 meaning.
    return min(1.0, max(0.0, float(u)))


VARIANTS = {
    "threshold": {"family": "threshold", "confirm": 0, "recover": 0},
    "threshold_persist": {"family": "threshold", "confirm": CONFIRM_S, "recover": RECOVER_S},
    "concern_level": {"family": "concern", "baseline": False, "trend": False},
    "concern_baseline": {"family": "concern", "baseline": True, "trend": False},
    "concern_trend": {"family": "concern", "baseline": True, "trend": True},
    "deficit_dose": {"family": "dose"},
}


def contiguous_mask(t, oxygen):
    valid = np.isfinite(t) & np.isfinite(oxygen)
    contiguous = np.r_[False, (np.diff(t) == 1) & valid[1:] & valid[:-1]]
    return valid, contiguous


def replay(t, oxygen, baseline, anchors, variant, auto_clear=False):
    """Causal replay of one session. Returns per-second concern, warning flag and alert id.
    auto_clear=False (device behaviour): an open alert stays latched until acknowledged outside this function.
    auto_clear=True (evaluation): it clears after RECOVER_S s at or below EXIT_SCORE."""
    cfg = VARIANTS[variant]
    start, full = anchors["start"], anchors["full"]
    midpoint = (start + full) / 2
    valid, contiguous = contiguous_mask(t, oxygen)
    n = len(t)
    concern = np.full(n, np.nan)
    warning = np.zeros(n, bool)
    alert_id = np.full(n, -1)
    window, state, level, slope, dose = [], None, None, 0.0, 0.0
    active, high_since, low_since, sequence, last_t = False, None, None, -1, None
    confirm = cfg.get("confirm", CONFIRM_S)
    recover = cfg.get("recover", RECOVER_S)
    for i in range(n):
        if not contiguous[i]:
            # A gap restarts smoothing/tracking and timers; an open alert stays latched.
            window, level, slope, last_t = [], None, 0.0, None
            high_since = low_since = None
        if not valid[i]:
            warning[i] = active
            alert_id[i] = sequence if active else -1
            continue
        x = float(oxygen[i])
        window = (window + [x])[-SMOOTH_S:]
        if len(window) < SMOOTH_S:
            warning[i] = active
            alert_id[i] = sequence if active else -1
            continue
        # same value as np.median, much faster on 3 values
        smoothed = float(statistics.median(window))
        dt = 1.0 if last_t is None else float(t[i] - last_t)
        last_t = t[i]
        if level is None:
            level, slope = smoothed, 0.0
        else:  # alpha-beta tracker on the smoothed signal
            predicted = level + slope * dt
            residual = smoothed - predicted
            level = predicted + ALPHA * residual
            slope = slope + BETA * residual / dt
        if cfg["family"] == "threshold":
            value = 1.0 if smoothed <= midpoint else 0.0
        elif cfg["family"] == "dose":
            dose = max(0.0, dose + max(0.0, start - smoothed) - 0.5)  # 0.5 pp/s drift allowance
            if smoothed >= start:
                # decay once back above start
                dose *= 0.8
            value = min(1.0, dose / ((start - full) * DOSE_SECONDS))
        else:
            severity = ramp((start - smoothed) / (start - full))
            if cfg["baseline"] and baseline is not None:
                # same anchors, person-relative
                shifted_start = start + (baseline - REFERENCE_RESTING_SPO2)
                severity = max(severity, ramp((shifted_start - smoothed) / (start - full)))
            if cfg["trend"] and slope < 0:
                projected = level + slope * TREND_HORIZON_S
                severity = max(severity, ramp((start - projected) / (start - full)))
            state = (
                severity
                if state is None
                else state + (1 - math.exp(-dt / TAU_S)) * (severity - state)
            )
            value = state
        concern[i] = value
        high, low = value >= ENTER_SCORE, value <= EXIT_SCORE
        if not active:
            low_since = None
            high_since = (t[i] if high_since is None else high_since) if high else None
            if high_since is not None and t[i] - high_since >= confirm:
                active, high_since, sequence = True, None, sequence + 1
        else:
            high_since = None
            low_since = (t[i] if low_since is None else low_since) if low else None
            if auto_clear and low_since is not None and t[i] - low_since >= recover:
                active, low_since = False, None
        warning[i] = active
        alert_id[i] = sequence if active else -1
    return {"concern": concern, "warning": warning, "alert_id": alert_id, "valid": valid}

### Kalman variants

`kalman_level` and `kalman_delay` replace the median and exponential filter with the fitted Kalman level. `kalman_delay` also advances a falling level by the known processing delay. `replay_candidate` dispatches to either replay.

In [ ]:
EVALUATION_VARIANTS = {
    **VARIANTS,
    "kalman_level": {"family": "kalman"},
    "kalman_delay": {"family": "kalman"},
}


def replay_kalman(t, oxygen, anchors, auto_clear=False, processing_delay_s=0.0):
    if (
        not np.isfinite(processing_delay_s)
        or not 0 <= processing_delay_s <= DEVICE_PROCESSING_DELAY_S
    ):
        raise ValueError("Projection must be a known processing delay between 0 and 1.995 seconds")
    t, oxygen = np.asarray(t, float), np.asarray(oxygen, float)
    valid, contiguous = contiguous_mask(t, oxygen)
    valid &= (oxygen > 0) & (oxygen <= 100)
    contiguous &= valid & np.r_[False, valid[:-1]]
    n = len(t)
    concern, levels, slopes = np.full(n, np.nan), np.full(n, np.nan), np.full(n, np.nan)
    warning, alert_id = np.zeros(n, bool), np.full(n, -1)
    active, high_since, low_since, sequence, level = False, None, None, -1, None
    for i in range(n):
        if not contiguous[i]:
            level, high_since, low_since = None, None, None
        if not valid[i]:
            warning[i], alert_id[i] = active, sequence if active else -1
            continue
        reading = float(oxygen[i])
        if level is None:
            level, slope, p00, p01, p11 = reading, 0.0, KALMAN_R, 0.0, 1.0
        else:
            level, slope, p00, p01, p11, _, _ = kalman_step(
                level, slope, p00, p01, p11, reading, KALMAN_Q, KALMAN_R
            )
        levels[i], slopes[i] = level, slope
        # A decline only advances the level by the known processing age; never extrapolate 15 s
        # ahead.
        aligned = level + min(slope, 0.0) * processing_delay_s
        value = ramp((anchors["start"] - aligned) / (anchors["start"] - anchors["full"]))
        concern[i] = value
        if not active:
            low_since = None
            high_since = (
                (t[i] if high_since is None else high_since) if value >= ENTER_SCORE else None
            )
            if high_since is not None and t[i] - high_since >= CONFIRM_S:
                active, high_since, sequence = True, None, sequence + 1
        else:
            high_since = None
            low_since = (t[i] if low_since is None else low_since) if value <= EXIT_SCORE else None
            if auto_clear and low_since is not None and t[i] - low_since >= RECOVER_S:
                active, low_since = False, None
        warning[i], alert_id[i] = active, sequence if active else -1
    return {
        "concern": concern,
        "warning": warning,
        "alert_id": alert_id,
        "valid": valid,
        "level": levels,
        "slope": slopes,
    }


def replay_candidate(
    t, oxygen, baseline, anchors, variant, auto_clear=False, processing_delay_s=0.0
):
    if variant.startswith("kalman_"):
        return replay_kalman(
            t, oxygen, anchors, auto_clear, processing_delay_s if variant == "kalman_delay" else 0.0
        )
    return replay(t, oxygen, baseline, anchors, variant, auto_clear)

### Behaviour checks

Deterministic assertions that every variant must pass before any evaluation runs.

In [ ]:
# ---- Behaviour checks (deterministic) ----
_t = np.arange(40.0)
_flat = np.full(40, 98.0)
for _v in EVALUATION_VARIANTS:
    assert not replay_candidate(_t, _flat, 98.0, ANCHOR_SETS["deep_breath_hold"], _v)[
        "warning"
    ].any(), _v
_low = np.r_[np.full(10, 98.0), np.full(30, 55.0)]
for _v in EVALUATION_VARIANTS:
    assert replay_candidate(_t, _low, 98.0, ANCHOR_SETS["deep_breath_hold"], _v)["warning"][-1], _v
_gap = np.r_[np.full(10, 98.0), np.full(10, 55.0), np.full(5, np.nan), np.full(8, 98.0)]
_r = replay_candidate(
    np.arange(33.0), _gap, 98.0, ANCHOR_SETS["deep_breath_hold"], "threshold_persist"
)
assert _r["warning"][22], "An alert must stay latched through missing data"
_recovered = np.r_[np.full(10, 98.0), np.full(15, 55.0), np.full(20, 98.0)]
for _v in EVALUATION_VARIANTS:
    assert replay_candidate(np.arange(45.0), _recovered, 98.0, ANCHOR_SETS["deep_breath_hold"], _v)[
        "warning"
    ][-1], f"{_v}: on the device an alert must stay on until acknowledged"
    assert not replay_candidate(
        np.arange(45.0), _recovered, 98.0, ANCHOR_SETS["deep_breath_hold"], _v, auto_clear=True
    )["warning"][-1], f"{_v}: evaluation mode clears after recovery"
# The person-relative branch must stay within the anchor set it belongs to: an 85% plateau is above
# the deep set's start even for a 100% baseline (shifted start 83%), so no non-trend deep variant
# may alarm on it.
_plateau = np.r_[np.full(10, 98.0), np.full(60, 85.0)]
for _v in ["threshold", "threshold_persist", "concern_level", "concern_baseline", "deficit_dose"]:
    assert not replay_candidate(
        np.arange(70.0), _plateau, 100.0, ANCHOR_SETS["deep_breath_hold"], _v
    )["warning"].any(), _v
print("Behaviour checks passed for", len(EVALUATION_VARIANTS), "variants.")

## Evaluation 1: real breath-hold sessions (no SpO₂-derived labels)

Each `(anchor set, variant)` policy runs on all 80 sessions. Because nobody lost consciousness, each **alert start** is a non-blackout alert. The table reports:
- alert starts per hour
- the fraction of time spent in warning
- the share of sessions with any alert, separately for breath-hold (AG) and normal breathing (CG)
- an exact Poisson 95% confidence interval on each rate. Each group has only about 2.9 hours, so zero alerts is still compatible with roughly one alert per hour. The interval treats alerts as independent, which understates the width when a few people produce most alerts.

The **AG-vs-CG AUROC** of each session's peak concern asks whether the index responds to breath-hold exposure at all. The group label comes from the protocol, not from SpO₂.

In [ ]:
def auroc(positive, negative):
    positive, negative = np.asarray(positive, float), np.asarray(negative, float)
    if not len(positive) or not len(negative):
        return np.nan
    greater = (positive[:, None] > negative[None, :]).mean()
    ties = (positive[:, None] == negative[None, :]).mean()
    return float(greater + 0.5 * ties)


session_rows = []
replays = {}
for anchor_name, anchors in ANCHOR_SETS.items():
    for variant in EVALUATION_VARIANTS:
        for s in sessions:
            r = replay_candidate(
                s["t"], s["oxygen"], s["baseline"], anchors, variant, auto_clear=True
            )
            replays[(anchor_name, variant, s["subject"])] = r
            starts = len(np.unique(r["alert_id"][r["warning"]]))
            session_rows.append(
                {
                    "anchors": anchor_name,
                    "variant": variant,
                    "subject": s["subject"],
                    "group": s["group"],
                    "valid_seconds": int(r["valid"].sum()),
                    "alert_starts": starts,
                    "warning_seconds": int((r["warning"] & r["valid"]).sum()),
                    "peak_concern": (
                        float(np.nanmax(r["concern"]))
                        if np.isfinite(r["concern"]).any()
                        else np.nan
                    ),
                }
            )
session_results = pd.DataFrame(session_rows)


def summarize(frame):
    hours = frame.valid_seconds.sum() / 3600
    lo, hi = poisson_rate_ci(int(frame.alert_starts.sum()), hours)
    return pd.Series(
        {
            "people": len(frame),
            "hours": hours,
            "alert_starts": int(frame.alert_starts.sum()),
            "alert_starts_per_hour": frame.alert_starts.sum() / hours,
            "rate_ci95_lo": lo,
            "rate_ci95_hi": hi,
            "warning_fraction": frame.warning_seconds.sum() / frame.valid_seconds.sum(),
            "sessions_with_alert": (frame.alert_starts > 0).mean(),
        }
    )


burden = (
    session_results.groupby(["anchors", "variant", "group"])
    .apply(summarize, include_groups=False)
    .unstack("group")
)
burden.columns = [f"{metric}_{group}" for metric, group in burden.columns]
burden["AG_vs_CG_peak_concern_AUROC"] = [
    auroc(
        g.loc[g.group.eq("AG"), "peak_concern"].fillna(0),
        g.loc[g.group.eq("CG"), "peak_concern"].fillna(0),
    )
    for _, g in session_results.groupby(["anchors", "variant"])
]
display(burden.round(3))
print("Every alert above is a non-blackout alert: all 80 participants remained conscious.")

## Evaluation 2: engineering latency (same-stream, not accuracy)

An *engineering event* is a run where SpO₂ is at or below the anchor midpoint for at least `ENGINEERING_EVENT_S` elapsed seconds. The event is defined from the same SpO₂ stream the rule reads, so this measures **reaction delay and debounce behaviour only**, not whether the rule detects anything physiologically meaningful. Expect few or no events under the deeper anchor sets: these breath-holds rarely go that low.

In [ ]:
def engineering_events(t, oxygen, level):
    valid, contiguous = contiguous_mask(t, oxygen)
    low = valid & (oxygen <= level)
    starts = np.flatnonzero(low & ~(np.r_[False, low[:-1]] & contiguous))
    ends = np.flatnonzero(low & ~(np.r_[low[1:], False] & np.r_[contiguous[1:], False]))
    return [(s, e) for s, e in zip(starts, ends) if t[e] - t[s] >= ENGINEERING_EVENT_S]


latency_rows = []
for anchor_name, anchors in ANCHOR_SETS.items():
    level = (anchors["start"] + anchors["full"]) / 2
    for s in sessions:
        events = engineering_events(s["t"], s["oxygen"], level)
        for variant in EVALUATION_VARIANTS:
            r = replays[(anchor_name, variant, s["subject"])]
            for lo, hi in events:
                inside = np.flatnonzero(r["warning"][lo : hi + 1])
                latency_rows.append(
                    {
                        "anchors": anchor_name,
                        "variant": variant,
                        "subject": s["subject"],
                        "group": s["group"],
                        "event_seconds": float(s["t"][hi] - s["t"][lo]),
                        "warned_during_event": bool(len(inside)),
                        "delay_s": float(inside[0]) if len(inside) else np.nan,
                        "warning_before_onset": bool(r["warning"][lo]),
                    }
                )
latency = pd.DataFrame(
    latency_rows,
    columns=[
        "anchors",
        "variant",
        "subject",
        "group",
        "event_seconds",
        "warned_during_event",
        "delay_s",
        "warning_before_onset",
    ],
)
if latency.empty:
    print("No engineering events under any anchor set.")
else:
    display(
        latency.groupby(["anchors", "variant"])
        .agg(
            events=("subject", "size"),
            people=("subject", "nunique"),
            warned=("warned_during_event", "mean"),
            median_delay_s=("delay_s", "median"),
            max_delay_s=("delay_s", "max"),
        )
        .round(2)
    )

## Evaluation 3: sensor noise and dropouts (seeded simulation on real traces)

The wearable's chest MAX30102 SpO₂ will be noisier than a Nonin finger oximeter, and it drops samples during motion. This section perturbs the real normal-breathing (CG) traces by:
- adding Gaussian noise (σ = 1, 2 or 3 percentage points)
- randomly removing 5% of seconds

Independent noise is the easy case: the 3-s median removes most of it. Real motion artefacts arrive in **bursts** and are **biased**. When noise dominates both PPG channels, the red/IR ratio tends to 1. The firmware's own Maxim calibration maps a ratio of 1 to **80.1%**, so a moving wearer's reading drifts towards about 80%. The `burst_artefact` column replaces 5% of seconds, in bursts averaging 8 s, with 80.1 ± 2%. The noise levels and burst statistics are assumptions, not a measured MAX30102 error model; only the 80.1% level comes from the device's calibration.

In [ ]:
rng = np.random.default_rng(SEED)
noise_rows = []
primary = ANCHOR_SETS[PRIMARY_ANCHORS]
for sigma in [0.0, 1.0, 2.0, 3.0]:
    for repeat in range(5 if sigma else 1):
        for s in sessions:
            if s["group"] != "CG":
                continue
            noisy = s["oxygen"] + rng.normal(0, sigma, len(s["oxygen"]))
            noisy[rng.random(len(noisy)) < 0.05] = np.nan
            noisy = np.clip(np.round(noisy), 0, 100)
            for anchor_name in ANCHOR_SETS:
                for variant in EVALUATION_VARIANTS:
                    r = replay_candidate(
                        s["t"],
                        noisy,
                        s["baseline"],
                        ANCHOR_SETS[anchor_name],
                        variant,
                        auto_clear=True,
                    )
                    noise_rows.append(
                        {
                            "scenario": f"sigma_{sigma:g}pp",
                            "anchors": anchor_name,
                            "variant": variant,
                            "alert_starts": len(np.unique(r["alert_id"][r["warning"]])),
                            "valid_seconds": int(r["valid"].sum()),
                        }
                    )

### Burst artefact at the Maxim R = 1 value (80.1%)

In [ ]:
def burst_mask(n, rng):
    """Two-state Markov bursts: mean length ARTIFACT_MEAN_BURST_S, long-run share ARTIFACT_FRACTION."""
    p_end = 1 / ARTIFACT_MEAN_BURST_S
    p_start = ARTIFACT_FRACTION * p_end / (1 - ARTIFACT_FRACTION)
    mask, on = np.zeros(n, bool), False
    for i in range(n):
        on = (rng.random() >= p_end) if on else (rng.random() < p_start)
        mask[i] = on
    return mask


for repeat in range(20):
    for s in sessions:
        if s["group"] != "CG":
            continue
        corrupted = s["oxygen"].copy()
        burst = burst_mask(len(corrupted), rng)
        corrupted[burst] = np.round(ARTIFACT_SPO2 + rng.normal(0, ARTIFACT_SD, burst.sum()))
        for anchor_name in ANCHOR_SETS:
            for variant in EVALUATION_VARIANTS:
                r = replay_candidate(
                    s["t"],
                    corrupted,
                    s["baseline"],
                    ANCHOR_SETS[anchor_name],
                    variant,
                    auto_clear=True,
                )
                noise_rows.append(
                    {
                        "scenario": "burst_artefact",
                        "anchors": anchor_name,
                        "variant": variant,
                        "alert_starts": len(np.unique(r["alert_id"][r["warning"]])),
                        "valid_seconds": int(r["valid"].sum()),
                    }
                )
noise = pd.DataFrame(noise_rows)
noise_table = (
    noise.groupby(["anchors", "variant", "scenario"])
    .apply(lambda g: g.alert_starts.sum() / (g.valid_seconds.sum() / 3600), include_groups=False)
    .rename("CG_alert_starts_per_hour")
    .unstack("scenario")
)
display(noise_table.round(2))
print(
    "Simulated repeats share the same 40 people, so these rates are not independent hours: read them as "
    "behaviour under each perturbation, not as estimates with narrow intervals."
)

## Evaluation 4: declared synthetic deep desaturations (behaviour only)

Synthetic trajectories stand in for the deep desaturations that no open dataset provides:
- start at the resting baseline (98%)
- hold for 20 s
- fall linearly at 0.2–2.0 %/s to a nadir between 85% and 45%
- hold for 15 s, then recover

The trajectory is the **arterial** saturation. Before the rule sees it, it passes through a declared sensor model:
- a pure delay of 0, 10, 20 or 40 s (`SENSOR_DELAY_S`) from arterial blood to the reported value
- the Maxim routine's 4-s averaging window
- rounding to whole percent at 1 Hz

The delay for a chest reflectance MAX30102 has not been measured. The sweep shows how much the answer depends on it. A controlled-desaturation dataset with arterial reference samples would be needed to measure it.

For each policy and delay, the table reports when the first warning occurs relative to the moment the **arterial** trajectory crosses the anchor set's `full` level. A negative value means the warning came earlier. These curves are **assumed shapes**, not physiology data, and the `full` level is a design anchor, not a blackout threshold. They show how the rule behaves, nothing more.

In [ ]:
def synthetic(nadir, rate, baseline=98.0, hold=20, nadir_hold=15, tail=60):
    """Arterial saturation trajectory (unrounded), 1 Hz."""
    fall = int(np.ceil((baseline - nadir) / rate))
    descent = baseline - rate * np.arange(1, fall + 1)
    trace = np.r_[
        np.full(hold, baseline),
        np.maximum(descent, nadir),
        np.full(nadir_hold, nadir),
        np.linspace(nadir, baseline, 20),
        np.full(tail, baseline),
    ]
    return np.arange(len(trace), dtype=float), trace


def sensed(arterial, delay_s, baseline=98.0):
    """Declared sensor model: pure delay, then a trailing DEVICE_WINDOW_S mean, then whole-percent rounding."""
    delayed = np.r_[np.full(int(delay_s), baseline), arterial][: len(arterial)]
    averaged = pd.Series(delayed).rolling(DEVICE_WINDOW_S, min_periods=1).mean().to_numpy()
    return np.round(averaged)


synthetic_rows = []
for nadir in [85, 75, 65, 55, 45]:
    for rate in [0.2, 0.5, 1.0, 2.0]:
        t, arterial = synthetic(nadir, rate)
        for delay in SENSOR_DELAY_S:
            reading = sensed(arterial, delay)
            for anchor_name, anchors in ANCHOR_SETS.items():
                # arterial reference, not the reading
                crossing = np.flatnonzero(arterial <= anchors["full"])
                for variant in EVALUATION_VARIANTS:
                    r = replay_candidate(
                        t,
                        reading,
                        98.0,
                        anchors,
                        variant,
                        auto_clear=True,
                        processing_delay_s=(DEVICE_WINDOW_S - 1) / 2,
                    )
                    first = np.flatnonzero(r["warning"])
                    synthetic_rows.append(
                        {
                            "nadir": nadir,
                            "rate_pp_per_s": rate,
                            "sensor_delay_s": delay,
                            "anchors": anchor_name,
                            "variant": variant,
                            "warned": bool(len(first)),
                            "reading_at_warning": (
                                float(reading[first[0]]) if len(first) else np.nan
                            ),
                            "arterial_at_warning": (
                                float(arterial[first[0]]) if len(first) else np.nan
                            ),
                            "warning_minus_arterial_full_crossing_s": (
                                float(first[0] - crossing[0])
                                if len(first) and len(crossing)
                                else np.nan
                            ),
                        }
                    )
synthetic_results = pd.DataFrame(synthetic_rows)
primary_rows = synthetic_results[
    synthetic_results.anchors.eq(PRIMARY_ANCHORS)
    & (synthetic_results.nadir <= ANCHOR_SETS[PRIMARY_ANCHORS]["full"])
]
print(
    f'{PRIMARY_ANCHORS}: warning time minus arterial crossing of {ANCHOR_SETS[PRIMARY_ANCHORS]["full"]:.0f}% (s); '
    "rows: fall rate, columns: sensor delay. Negative = before the crossing."
)
for variant in EVALUATION_VARIANTS:
    print(f"\n{variant}")
    display(
        primary_rows[primary_rows.variant.eq(variant)]
        .pivot_table(
            index=["nadir", "rate_pp_per_s"],
            columns="sensor_delay_s",
            values="warning_minus_arterial_full_crossing_s",
        )
        .round(0)
    )
shallow = synthetic_results[
    synthetic_results.anchors.eq(PRIMARY_ANCHORS)
    & (synthetic_results.nadir > ANCHOR_SETS[PRIMARY_ANCHORS]["start"])
]
print(
    "Primary-anchor alerts on dips that never go below the start anchor:",
    shallow.groupby("variant").warned.sum().to_dict(),
)

## Evaluation 5: real controlled desaturations against independent oximeters (UW/UCSD, open)

**Data.** [Hoffman et al., *npj Digital Medicine* 2022](https://www.nature.com/articles/s41746-022-00665-y); [repository](https://github.com/ubicomplab/oximetry-phone-cam-data), MIT licence, pinned to commit `c483ae8`.
- 6 awake adults breathed a controlled low-oxygen gas mixture for about 15 minutes.
- Four clinical finger oximeters (3900P, two Nellcor N-600X, Masimo Radical-7) recorded SpO₂ and pulse rate at 1 Hz.
- The lowest readings were 61–77%. Nobody lost consciousness.

**Method (no invented labels).** The rule runs on one oximeter's trace at a time. The reference is the **per-second median of the other three oximeters**: independent instruments, not the stream the rule reads.
- A **reference crossing** is the first second the reference stays at or below an anchor set's midpoint for ≥ 5 s.
- The table reports whether and when each variant warned, relative to that crossing.
- **Agreement:** each device's bias and spread against the reference, by saturation band.
- **Relative delay:** cross-correlation of second-to-second changes, within ±30 s.

**Limits.**
- 6 people only.
- The desaturations are slow (minutes, not the seconds of an ascent).
- All sensors are finger oximeters, so the reference shares their site delay.
- No arterial blood samples; OpenOximetry provides those (next section).
- Subjects were *breathing* low-oxygen gas, so heart rate rises. In a breath-hold it falls, so heart-rate behaviour here does not transfer to swimming.
- The baseline is each device's own median over the first 60 s, before the gas change. This is measured, but on the same device.

In [ ]:
UW_COMMIT = "c483ae8fe4cd32b0b9ce412d07299b05279fc58f"
UW_FILES = {
    "100001.csv": "1b14056cc6a26781df7dce95ded359ca",
    "100002.csv": "59ad06c0c20a212584c94fe074c8d7a4",
    "100003.csv": "51788eed39fc1990f1b378e36dad19b6",
    "100004.csv": "6f17c15e32b1ff52e4d3b9c320fb9a48",
    "100005.csv": "15236e2672760cc60f8b1df486d96618",
    "100006.csv": "04d2a34d000d04de94cc6bb72281ecb3",
}
UW_DEVICES = {
    "SpO2 1": "3900P",
    "SpO2 2": "Nellcor N-600X (a)",
    "SpO2 4": "Nellcor N-600X (b)",
    "SpO2 5": "Masimo Radical-7",
}
UW_HOLD_S, UW_BASELINE_S, UW_MAX_LAG_S = 5, 60, 30


def load_uw(path):
    frame = pd.read_csv(path)
    frame.columns = [str(c).strip().lstrip("﻿") for c in frame.columns]
    # 'Time' or unnamed
    clock = pd.to_timedelta(frame[frame.columns[0]].astype(str).str.strip(), errors="coerce")
    # drops 'Collection Halted'
    frame, clock = frame[clock.notna()], clock[clock.notna()]
    seconds = (clock - clock.iloc[0]).dt.total_seconds().round().to_numpy()
    if (np.diff(seconds) <= 0).any():
        raise ValueError(f"{path.name}: clock is not increasing")

    def column(name):
        v = pd.to_numeric(frame[name], errors="coerce").to_numpy(float)
        return np.where((v > 0) & (v <= 100) if name.startswith("SpO2") else (v > 0), v, np.nan)

    return seconds, {d: column(d) for d in UW_DEVICES}, column("Pulse 5")

Download the six pinned CSV files and read each device's 1 Hz trace.

In [ ]:
uw = {}
for name, digest in UW_FILES.items():
    path = acquire(
        name,
        f"https://raw.githubusercontent.com/ubicomplab/oximetry-phone-cam-data/{UW_COMMIT}/data/gt/{name}",
        digest,
    )
    uw[name.split(".")[0]] = load_uw(path)

The reference for each device is the per-second median of the other three. Helpers find the reference crossing and the relative lag.

In [ ]:
def consensus(traces, exclude):
    others = np.vstack([v for d, v in traces.items() if d != exclude])
    enough = np.isfinite(others).sum(0) >= 2
    with np.errstate(all="ignore"):
        return np.where(enough, np.nanmedian(others, 0), np.nan)


def first_run(mask, t, hold):
    run_start = None
    for i in range(len(mask)):
        if mask[i]:
            run_start = i if run_start is None else run_start
            if t[i] - t[run_start] >= hold:
                return run_start
        else:
            run_start = None
    return None


def relative_lag(x, ref):
    # Seconds by which x trails ref (positive = x later), from cross-correlating 1-s changes.
    dx, dr = np.nan_to_num(np.diff(x)), np.nan_to_num(np.diff(ref))
    lags = range(-UW_MAX_LAG_S, UW_MAX_LAG_S + 1)
    score = [
        np.dot(dx[max(0, k) : len(dx) + min(0, k)], dr[max(0, -k) : len(dr) - max(0, k)])
        for k in lags
    ]
    return list(lags)[int(np.argmax(score))]

Replay every variant on each device and time it against the reference.

In [ ]:
agreement_rows, timing_rows, hr_rows = [], [], []
for subject, (t, traces, pulse) in uw.items():
    for device, label in UW_DEVICES.items():
        reading, ref = traces[device], consensus(traces, device)
        both = np.isfinite(reading) & np.isfinite(ref)
        for band, lo, hi in [
            ("ref >= 90", 90, 101),
            ("80 <= ref < 90", 80, 90),
            ("ref < 80", 0, 80),
        ]:
            take = both & (ref >= lo) & (ref < hi)
            if take.sum():
                err = reading[take] - ref[take]
                agreement_rows.append(
                    {
                        "subject": subject,
                        "device": label,
                        "band": band,
                        "seconds": int(take.sum()),
                        "bias": err.mean(),
                        "sd": err.std(),
                        "abs_max": np.abs(err).max(),
                    }
                )
        agreement_rows.append(
            {
                "subject": subject,
                "device": label,
                "band": "relative lag (s)",
                "seconds": int(both.sum()),
                "bias": relative_lag(reading, ref),
                "sd": np.nan,
                "abs_max": np.nan,
            }
        )
        baseline = float(np.nanmedian(reading[:UW_BASELINE_S]))
        for anchor_name, anchors in ANCHOR_SETS.items():
            midpoint = (anchors["start"] + anchors["full"]) / 2
            onset = first_run(np.nan_to_num(ref, nan=101) <= midpoint, t, UW_HOLD_S)
            for variant in EVALUATION_VARIANTS:
                r = replay_candidate(t, reading, baseline, anchors, variant, auto_clear=True)
                first = np.flatnonzero(r["warning"])
                timing_rows.append(
                    {
                        "subject": subject,
                        "device": label,
                        "anchors": anchor_name,
                        "variant": variant,
                        "reference_crossing": onset is not None,
                        "warned": bool(len(first)),
                        "warning_minus_reference_crossing_s": (
                            float(t[first[0]] - t[onset])
                            if len(first) and onset is not None
                            else np.nan
                        ),
                        "reading_at_warning": float(reading[first[0]]) if len(first) else np.nan,
                        "reference_at_warning": float(ref[first[0]]) if len(first) else np.nan,
                        "alert_starts": len(np.unique(r["alert_id"][r["warning"]])),
                    }
                )
    ref_all = np.nanmedian(np.vstack(list(traces.values())), 0)
    for band, lo, hi in [(">= 95", 95, 101), ("90-95", 90, 95), ("80-90", 80, 90), ("< 80", 0, 80)]:
        take = np.isfinite(pulse) & (ref_all >= lo) & (ref_all < hi)
        if take.sum():
            hr_rows.append(
                {
                    "subject": subject,
                    "spo2_band": band,
                    "seconds": int(take.sum()),
                    "median_pulse_bpm": float(np.median(pulse[take])),
                }
            )

### Results

In [ ]:
uw_agreement, uw_timing, uw_hr = (
    pd.DataFrame(agreement_rows),
    pd.DataFrame(timing_rows),
    pd.DataFrame(hr_rows),
)
print(
    "Lowest reading per subject:",
    {s: float(np.nanmin(np.vstack(list(v[1].values())))) for s, v in uw.items()},
)
print(
    "\nDevice agreement with the median of the other three (percentage points; lag in s, positive = later):"
)
display(
    uw_agreement.groupby(["device", "band"])[["seconds", "bias", "sd", "abs_max"]]
    .agg({"seconds": "sum", "bias": "median", "sd": "median", "abs_max": "max"})
    .round(2)
)
summary = (
    uw_timing[uw_timing.reference_crossing]
    .groupby(["anchors", "variant"])
    .agg(
        device_traces=("warned", "size"),
        warned=("warned", "sum"),
        median_s=("warning_minus_reference_crossing_s", "median"),
        max_s=("warning_minus_reference_crossing_s", "max"),
    )
)
print(
    "\nTraces whose reference crossed each anchor midpoint: did the rule on the held-out device warn, and when?"
)
display(summary.round(1))
quiet = (
    uw_timing[~uw_timing.reference_crossing]
    .groupby(["anchors", "variant"])
    .agg(device_traces=("warned", "size"), warned_without_reference_crossing=("warned", "sum"))
)
print("\nTraces whose reference never crossed the midpoint: warnings anyway (device disagreement):")
display(quiet)
print("\nPulse rate by SpO2 band (breathing low-oxygen gas: expect a RISE, unlike a breath-hold):")
display(uw_hr.groupby("spo2_band").median_pulse_bpm.median().round(0))

## Optional: OpenOximetry arterial reference (restricted access, needs your PhysioNet login)

[OpenOximetry](https://physionet.org/content/openox-repo/1.1.1/) pairs pulse-oximeter readings with **arterial blood-gas SaO₂**, the gold standard, at 70–100%. Access requires a PhysioNet account and the signed data use agreement ([sign here](https://physionet.org/sign-dua/openox-repo/1.1.1/)).

This cell downloads **only the tables** (about 21 MB) instead of the 3.2 GB waveforms:

```
physionet download openox-repo --version 1.1.1 --include "*.csv" --exclude "waveforms/*"
```

- **Password:** typed into a hidden prompt (or read from the Colab secret `PHYSIONET_PASSWORD`). It is passed only to that command, never printed or saved.
- **Default:** off. Set `OPENOX_ENABLE = True` (or `KB_OPENOX=1`) to run it.
- **Output:** files, row counts and column names only. The reference analysis is written once the column names are confirmed against the data dictionary.

**Data-use note.** Keep the data in your private Drive. Don't paste rows into chats or online AI tools; share only aggregate results.

In [ ]:
OPENOX_ENABLE = env_flag("KB_OPENOX", False)
OPENOX_USER = os.getenv("PHYSIONET_USERNAME", "mitcheltu")
OPENOX_DIR = Path(os.getenv("OPENOX_DIR", str(BASE / "restricted" / "openox")))


def physionet_password():
    try:
        from google.colab import userdata  # Colab secret, if the user created one

        value = userdata.get("PHYSIONET_PASSWORD")
        if value:
            return value
    except Exception:
        pass
    import getpass

    return getpass.getpass(f"PhysioNet password for {OPENOX_USER}: ")


if not OPENOX_ENABLE:
    print("OpenOximetry skipped (set OPENOX_ENABLE = True after signing the data use agreement).")
else:
    import subprocess

    tables = sorted(OPENOX_DIR.rglob("*.csv")) if OPENOX_DIR.exists() else []
    if not tables:
        if importlib.util.find_spec("physionet") is None:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", "physionet==0.2.1"], check=True
            )
        OPENOX_DIR.mkdir(parents=True, exist_ok=True)
        env = {
            **os.environ,
            "PHYSIONET_USERNAME": OPENOX_USER,
            "PHYSIONET_PASSWORD": physionet_password(),
        }
        import shutil

        cli = shutil.which("physionet") or "physionet"  # CLI installed by the physionet package
        done = subprocess.run(
            [
                cli,
                "download",
                "openox-repo",
                "--version",
                "1.1.1",
                "--include",
                "*.csv",
                "--exclude",
                "waveforms/*",
                "--output",
                str(OPENOX_DIR),
            ],
            env=env,
            capture_output=True,
            text=True,
        )
        del env
        if done.returncode != 0:
            raise RuntimeError(
                "PhysioNet download failed (check the signed agreement and password): "
                + done.stderr[-500:]
            )
        tables = sorted(OPENOX_DIR.rglob("*.csv"))
    for table in tables:
        columns = pd.read_csv(table, nrows=0).columns.tolist()
        rows = sum(1 for _ in open(table, encoding="utf-8", errors="replace")) - 1
        print(f"{table.name}: {rows:,} rows; columns: {columns}")

## Figures

### Breath-hold support: mean SpO₂ and session minima

In [ ]:
def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURES / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for group, color in [("AG", "tab:blue"), ("CG", "tab:gray")]:
    stack = np.array([s["oxygen"] for s in sessions if s["group"] == group])
    axes[0].plot(np.nanmean(stack, 0), color=color, label=f"{group} mean")
    axes[0].fill_between(
        np.arange(360),
        np.nanpercentile(stack, 5, 0),
        np.nanpercentile(stack, 95, 0),
        color=color,
        alpha=0.15,
    )
    axes[1].hist(
        audit.loc[audit.group.eq(group), "min_spo2"],
        bins=np.arange(80, 101),
        alpha=0.6,
        color=color,
        label=group,
    )
for name, anchors in ANCHOR_SETS.items():
    axes[1].axvline((anchors["start"] + anchors["full"]) / 2, ls=":", color="black")
axes[0].set(
    xlabel="Session second",
    ylabel="SpO2 (%)",
    title="Breath-hold (AG) vs normal breathing (CG): mean, 5-95%",
)
axes[1].set(
    xlabel="Lowest SpO2 in session (%)",
    ylabel="People",
    title="Session minimum vs anchor midpoints (dotted)",
)
axes[0].legend()
axes[1].legend()
save(fig, "01_breath_hold_support")

### Alert burden by anchor set and variant

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, (anchor_name, block) in zip(axes, burden.groupby(level="anchors")):
    block = block.droplevel("anchors")
    x = np.arange(len(block))
    for offset, group, label in [
        (-0.2, "AG", "Breath-hold (AG)"),
        (0.2, "CG", "Normal breathing (CG)"),
    ]:
        rate = block[f"alert_starts_per_hour_{group}"]
        err = np.vstack(
            [rate - block[f"rate_ci95_lo_{group}"], block[f"rate_ci95_hi_{group}"] - rate]
        )
        ax.bar(x + offset, rate, 0.4, yerr=err, capsize=2, label=label)
    ax.set_xticks(x, block.index, rotation=45, ha="right")
    ax.set_title(anchor_name)
axes[0].set_ylabel("Non-blackout alert starts / hour (95% CI)")
axes[0].legend()
save(fig, "02_alert_burden")

### Replay of the lowest-SpO₂ breath-hold session

In [ ]:
example = next(
    s for s in sorted(sessions, key=lambda s: np.nanmin(s["oxygen"])) if s["group"] == "AG"
)
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(example["t"], example["oxygen"], color="tab:blue")
for name, anchors in ANCHOR_SETS.items():
    axes[0].axhline((anchors["start"] + anchors["full"]) / 2, ls=":", color="gray")
axes[0].set(ylabel="SpO2 (%)", title=f"{example['subject']}: lowest-SpO2 breath-hold session")
for j, anchor_name in enumerate(ANCHOR_SETS):
    r = replays[(anchor_name, "concern_trend", example["subject"])]
    axes[1].plot(example["t"], 100 * r["concern"], label=anchor_name)
    axes[2].step(
        example["t"], r["warning"].astype(float) + 1.2 * j, where="post", label=anchor_name
    )
axes[1].axhline(100 * ENTER_SCORE, ls="--", color="black")
axes[1].set(ylabel="concern_trend (0-100)")
axes[2].set(ylabel="Warning (offset)", xlabel="Session second")
axes[1].legend(fontsize=8)
save(fig, "03_session_replay")

### Synthetic timing by sensor delay

In [ ]:
fig, axes = plt.subplots(
    1, len(SENSOR_DELAY_S), figsize=(4 * len(SENSOR_DELAY_S), 4.2), sharey=True
)
for ax, delay in zip(axes, SENSOR_DELAY_S):
    for variant in EVALUATION_VARIANTS:
        block = primary_rows[
            primary_rows.variant.eq(variant) & primary_rows.sensor_delay_s.eq(delay)
        ]
        ax.plot(
            block.rate_pp_per_s + 0.02 * list(EVALUATION_VARIANTS).index(variant),
            block.warning_minus_arterial_full_crossing_s,
            "o",
            ms=4,
            label=variant,
        )
    ax.axhline(0, color="black", lw=0.8)
    ax.set(xscale="log", xlabel="Synthetic fall rate (pp/s)", title=f"sensor delay {delay} s")
axes[0].set_ylabel("Warning - arterial full-anchor crossing (s)")
axes[0].legend(fontsize=7)
fig.suptitle(
    f"Synthetic deep desaturations, {PRIMARY_ANCHORS} anchors, declared sensor model (behaviour check)"
)
save(fig, "04_synthetic_timing")

### Real controlled desaturations (UW/UCSD)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 6.5), sharey=True)
for ax, (subject, (t, traces, _)) in zip(axes.flat, uw.items()):
    for device, label in UW_DEVICES.items():
        ax.plot(t / 60, traces[device], lw=0.8, label=label)
    for name, anchors in ANCHOR_SETS.items():
        ax.axhline((anchors["start"] + anchors["full"]) / 2, ls=":", color="gray", lw=0.8)
    ax.set(title=f"UW/UCSD {subject}", xlabel="minutes", ylim=(55, 101))
axes[0, 0].set_ylabel("SpO2 (%)")
axes[1, 0].set_ylabel("SpO2 (%)")
axes[0, 0].legend(fontsize=7)
fig.suptitle(
    "Real controlled desaturations: four clinical oximeters; dotted = anchor midpoints (90, 82.5, 70)"
)
save(fig, "05_real_desaturations")

## Export

The exported files are:
- the frozen configuration (`deployable: false`)
- all result tables
- an importable `replay_reference.py` containing the exact replay code

A firmware port must keep the state between updates. Replaying only a short rolling window would reset the latch and the timers. The port must use `auto_clear=False` and clear an alert only on a physical acknowledgement.

In [ ]:
import inspect

configuration = {
    "purpose": "oxygen concern rule; breath-hold evaluation of alert burden and behaviour",
    "deployable": False,
    "trained_on_outcomes": False,
    "blackout_probability": False,
    "anchor_sets": ANCHOR_SETS,
    "primary_anchors": PRIMARY_ANCHORS,
    "enter_score": ENTER_SCORE,
    "exit_score": EXIT_SCORE,
    "smooth_s": SMOOTH_S,
    "tau_s": TAU_S,
    "confirm_s": CONFIRM_S,
    "recover_s": RECOVER_S,
    "reference_resting_spo2": REFERENCE_RESTING_SPO2,
    "person_relative_branch": "anchor set shifted by (resting SpO2 - reference); max with absolute map",
    "alert_clearing": "device: latched until acknowledged (auto_clear=False); evaluations: auto_clear=True",
    "synthetic_sensor_model": {"delay_s": SENSOR_DELAY_S, "device_window_s": DEVICE_WINDOW_S},
    "burst_artefact": {
        "spo2": ARTIFACT_SPO2,
        "sd": ARTIFACT_SD,
        "fraction": ARTIFACT_FRACTION,
        "mean_burst_s": ARTIFACT_MEAN_BURST_S,
    },
    "trend_horizon_s": TREND_HORIZON_S,
    "alpha_beta": [ALPHA, BETA],
    "dose_seconds": DOSE_SECONDS,
    "variants": EVALUATION_VARIANTS,
    "input_rate_hz": 1,
    "baseline_source": "Resting_SpO2 recorded by the study before each session (STA_BVP sheet)",
    "limitations": [
        "dry supine breath-holds, not immersion",
        "finger Nonin oximeter, not chest MAX30102",
        "lowest observed SpO2 about 83%: real deep data are clinical oximeters without arterial reference",
        "no blackout outcomes; anchors are unvalidated",
        "sensor delay of the chest MAX30102 unmeasured: synthetic timing is a sweep",
        "Maxim calibration curve is generic, not calibrated for this sensor/site below ~90%",
    ],
    "version": "v5 (2026-11-16)",
    "kalman_fit": KALMAN_FIT,
    "anchor_derivation": ANCHOR_AUDIT,
    "kalman_projection": {
        "device_nominal_centre_age_s": DEVICE_PROCESSING_DELAY_S,
        "clinical_streams_s": 0,
        "synthetic_four_point_mean_s": (DEVICE_WINDOW_S - 1) / 2,
        "unknown_tissue_delay_compensated": False,
    },
    "real_desaturation_reference": {
        "dataset": "UW/UCSD controlled hypoxemia",
        "commit": UW_COMMIT,
        "reference": "median of the other three oximeters (not arterial)",
    },
    "versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
}
(OUT / "concern_configuration.json").write_text(
    json.dumps(configuration, indent=2), encoding="utf-8"
)
session_results.to_csv(OUT / "session_results.csv", index=False)
burden.to_csv(OUT / "alert_burden.csv")
latency.to_csv(OUT / "engineering_latency.csv", index=False)
noise_table.to_csv(OUT / "noise_robustness.csv")
synthetic_results.to_csv(OUT / "synthetic_behaviour.csv", index=False)
audit.to_csv(OUT / "participant_audit.csv", index=False)
uw_agreement.to_csv(OUT / "uw_device_agreement.csv", index=False)
uw_timing.to_csv(OUT / "uw_real_desaturation_timing.csv", index=False)
uw_hr.to_csv(OUT / "uw_pulse_by_spo2.csv", index=False)

### `replay_reference.py`

The exact replay code and constants as an importable module, checked against the notebook's own replays.

In [ ]:
constants = [
    "ANCHOR_SETS",
    "ENTER_SCORE",
    "EXIT_SCORE",
    "SMOOTH_S",
    "TAU_S",
    "CONFIRM_S",
    "RECOVER_S",
    "REFERENCE_RESTING_SPO2",
    "TREND_HORIZON_S",
    "ALPHA",
    "BETA",
    "DOSE_SECONDS",
    "VARIANTS",
    "EVALUATION_VARIANTS",
    "KALMAN_Q",
    "KALMAN_R",
    "DEVICE_PROCESSING_DELAY_S",
]
source = "# Frozen oxygen concern replay; research reference, not a validated controller.\nimport math\nimport statistics\nimport numpy as np\n\n"
source += "".join(f"{name} = {globals()[name]!r}\n" for name in constants) + "\n"
source += "\n\n".join(
    inspect.getsource(f)
    for f in [ramp, contiguous_mask, replay, kalman_step, replay_kalman, replay_candidate]
)
(OUT / "replay_reference.py").write_text(source, encoding="utf-8")
spec = importlib.util.spec_from_file_location("replay_reference", OUT / "replay_reference.py")
exported = importlib.util.module_from_spec(spec)
spec.loader.exec_module(exported)
check = sessions[0]
for variant in EVALUATION_VARIANTS:
    a = exported.replay_candidate(
        check["t"],
        check["oxygen"],
        check["baseline"],
        ANCHOR_SETS[PRIMARY_ANCHORS],
        variant,
        auto_clear=True,
    )
    b = replays[(PRIMARY_ANCHORS, variant, check["subject"])]
    assert np.array_equal(a["warning"], b["warning"]) and np.allclose(
        a["concern"], b["concern"], equal_nan=True
    )
print(
    "Exported replay reproduces the notebook. Files:",
    sorted(p.name for p in OUT.iterdir() if p.is_file()),
)

### Kalman fit, anchor audit and held-out controls

In [ ]:
(OUT / "kalman_fit.json").write_text(json.dumps(KALMAN_FIT, indent=2), encoding="utf-8")
(OUT / "anchor_audit.json").write_text(json.dumps(ANCHOR_AUDIT, indent=2), encoding="utf-8")
session_results.assign(calibration=lambda d: d.subject.isin(KALMAN_FIT_IDS)).to_csv(
    OUT / "session_results_with_fit_split.csv", index=False
)
# Separate control participants excluded from parameter fitting; still an inspected development
# comparison.
held = session_results[session_results.subject.isin(KALMAN_CHECK_IDS)]
held.groupby(["anchors", "variant"]).apply(summarize, include_groups=False).to_csv(
    OUT / "heldout_control_burden.csv"
)

## Reading the results

- **Alert burden** on breath-holds and normal breathing is the main real-data result. The clinical anchors (95→85%) will alert on ordinary breath-holds. In water, every swimmer holds their breath, so treat that burden as a nuisance-alarm estimate. The deep anchors should stay quiet here, but that also means **this dataset cannot show they fire in time**.
- **The AG-vs-CG AUROC** shows whether the index tracks breath-hold exposure at all. It does not show that it tracks blackout risk.
- **Synthetic timing** shows when each variant warns relative to the *arterial* crossing of the `full` anchor, for assumed fall speeds and sensor delays. Read across the delay columns: if the sign flips between 0 and 20 s, the rule's usefulness depends on a sensor property nobody has measured yet. Fast falls during ascent are the hard case. In practice, this rule can only detect a problem *after* it starts, so it should be judged on time-to-alarm, not on forecasting.
- **Real desaturations (Evaluation 5)** are the only real check of the deep anchors so far. Read the held-out-device delays against the synthetic sensor-delay sweep. Device disagreement of a few points at 70% is normal and sets a floor on how precise any anchor can be.
- **Burst artefact** shows what happens when motion drives the device's reading towards its 80.1% calibration point. Any variant that fires here will fire while the wearer is swimming hard.
- **Before use on the wearable:** validate chest MAX30102 SpO₂ against a reference oximeter in the 60–100% range, in water. Then obtain breath-hold recordings that reach deep desaturation, with outcome observation, and evaluate the frozen rule on them.